In [0]:
storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(scope = "aml-scope", key = "storage-access-key")

spark.conf.set("fs.azure.account.key.stbankamldev.blob.core.windows.net",storage_key)

df_rolling_transaction = spark.read.option("header","true").option("inferSchema","true").csv("wasbs://raw@stbankamldev.blob.core.windows.net/rolling_24h_transactions_v2.csv")


df_rolling_transaction.show()


In [0]:
df_rolling_transaction.printSchema()

In [0]:
from pyspark.sql.functions import col, lower, trim, upper , date_format, unix_timestamp
from datetime import datetime
df_rolling_trans_clean = df_rolling_transaction \
.withColumn("clean_counterparty_name" , lower(trim(col("counterparty_name")))) \
.withColumn("clean_counterparty_country" , upper(trim(col("counterparty_country")))) \
.withColumn("clean_transaction_type" ,  upper(trim(col("transaction_type")))) \
.withColumn("clean_channel", upper(trim(col("channel")))) \
.withColumn("clean_transaction_category" , upper(trim(col("transaction_category")))) \
.withColumn("clean_transaction_subcategory", upper(trim(col("transaction_subcategory")))) \
.withColumn("transaction_unix_time", unix_timestamp("transaction_timestamp")) \
.select("transaction_id","transaction_timestamp" ,"transaction_unix_time", "account_id","customer_id","counterparty_account_id", "clean_counterparty_name",  "clean_counterparty_country",  "transaction_amount","clean_transaction_type","clean_channel","clean_transaction_category","clean_transaction_subcategory")
 
#df_rolling_trans_clean.show(5)
df_rolling_trans_clean.write.mode("overwrite").parquet("wasbs://silver@stbankamldev.blob.core.windows.net/rolling_trans_clean.parquet")


storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(scope = "aml-scope", key = "storage-access-key")

spark.conf.set("fs.azure.account.key.stbankamldev.blob.core.windows.net",storage_key)

df_rolling_trans_clean_rd = spark.read.parquet("wasbs://silver@stbankamldev.blob.core.windows.net/rolling_trans_clean.parquet")
df_rolling_trans_clean_rd.show(5)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, count, col, when


window_spec = Window.partitionBy("customer_id").orderBy("transaction_unix_time").rangeBetween(-86400 , 0)

df_rolling_trans_trf = df_rolling_trans_clean_rd \
.withColumn(
    "rolling_24h_amount",
    sum("transaction_amount").over(window_spec)
) \
.withColumn(
    "rolling_24h_count",
    count("*").over(window_spec)
) \
.withColumn("rolling_24h_alert_flag" , when(col("rolling_24h_amount") >= 10000, "1").otherwise(0)) \
.withColumn("alert_reason", when(col("rolling_24h_amount") >= 10000, "ROLLING_24H_THRESHOLD_EXCEEDED").otherwise("None"))


df_rolling_trans_trf.show(5)

df_rolling_trans_trf.write.mode("overwrite").parquet("wasbs://gold@stbankamldev.blob.core.windows.net/rolling_24h_transaction_monitoring.parquet")

df_rolling_trans_trf_rd = spark.read.parquet("wasbs://gold@stbankamldev.blob.core.windows.net/rolling_24h_transaction_monitoring.parquet")


df_rolling_trans_trf_rd.show(5)


